# 02 - Data Understanding and Quality Checks
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Inspecting Portfolio Data Health
Before cleaning or transforming financial data, we need a clear look at its overall state:
1. Are column types appropriate (numerical vs. text)?
2. Where are the missing values, and how extensive are they?
3. Are there duplicate borrower entries?
4. How do our core financial variables look?
   - `revenue`: Annual income
   - `loan_amnt`: Funded loan amount
   - `dti_n`: Debt-to-income ratio (monthly debt payments divided by gross monthly income)
   - `fico_n`: Credit score at origination
   - `Default`: Repayment outcome (1 = Defaulted, 0 = Fully Paid)


In [ ]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw/loans_granting_model_dataset.csv")
if not raw_path.exists():
    raw_path = Path("data/raw/loans_granting_model_dataset.csv")

df = pd.read_csv(raw_path, nrows=25000)
print(f"Loaded {len(df):,} borrower records.")


---
## 1. Column Types & Non-Null Coverage
Using `.info()` gives us a quick snapshot of memory usage, types, and populated fields.


In [ ]:
df.info()


---
## 2. Missing Value Analysis
Missing values can break clustering distance calculations. Let's inspect where nulls appear.


In [ ]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_table = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percent.round(2)
})

missing_table[missing_table['Missing Values'] > 0]


---
## 3. Duplicate Records Check
Duplicate entries can distort cluster sizing and bias default calculations.


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate records found: {duplicate_count}")


---
## 4. Summary Statistics for Financial Variables
Using `.describe()` gives us the median, spread, minimum, and maximum across our loan variables.


In [ ]:
key_cols = ['revenue', 'loan_amnt', 'dti_n', 'fico_n', 'experience_c', 'Default']
existing_cols = [c for c in key_cols if c in df.columns]

df[existing_cols].describe().round(2).T


---
## 5. Portfolio Default Rate Baseline
What is the natural default rate across our historical borrower pool?


In [ ]:
default_counts = df['Default'].value_counts()
default_pct = df['Default'].value_counts(normalize=True) * 100

print("Default Counts:")
print(default_counts)
print("\nDefault Rate Distribution:")
print(default_pct.round(2).astype(str) + "%")


---
## Summary of Data Diagnostics
- The historical portfolio default rate sits at roughly 18-20%.
- A few financial fields contain missing values that need imputation.
- Cryptic column names (`dti_n`, `fico_n`, `revenue`) should be converted to clean industry terms.
- Up next in **Notebook 03**: We clean the data, impute missing values, and filter out corrupt records.
